In [1]:
import zipfile
import os

zip_path = "archive.zip"   # 👈 change this to your zip filename
extract_path = "data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))


Extracted files: ['personal_expense_classification.csv']


In [2]:
import os
os.listdir("data")


['personal_expense_classification.csv']

In [3]:
import pandas as pd

df = pd.read_csv("data/personal_expense_classification.csv")
df.head()


,expense_id,amount,merchant,description,category
0,EXP1,78.04,Amazon,fuel purchase,shopping
1,EXP2,190.39,Apple Store,online shopping,technology
2,EXP3,147.74,McDonald's,app purchase,food
3,EXP4,121.74,Starbucks,meal,food
4,EXP5,35.42,Netflix,fuel purchase,entertainment


In [4]:
df.columns


Index(['expense_id', 'amount', 'merchant', 'description', 'category'], dtype='object')

In [5]:
import pandas as pd

df = pd.read_csv("data/personal_expense_classification.csv")

# Rename columns to standard names
df = df.rename(columns={
    "description": "Description",
    "merchant": "Merchant",
    "amount": "Amount",
    "category": "Category"
})

# Combine merchant + description into one text column
df["Text"] = (
    df["Merchant"].astype(str) + " " + df["Description"].astype(str)
).str.lower()

# Keep only required columns
df = df[["Text", "Amount", "Category"]]
df.dropna(inplace=True)

df.head()


,Text,Amount,Category
0,amazon fuel purchase,78.04,shopping
1,apple store online shopping,190.39,technology
2,mcdonald's app purchase,147.74,food
3,starbucks meal,121.74,food
4,netflix fuel purchase,35.42,entertainment


In [6]:
from sklearn.model_selection import train_test_split

X = df[["Text", "Amount"]]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(stop_words="english"), "Text"),
    ("num", StandardScaler(), ["Amount"])
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])


In [8]:
model.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(stop_words='english'),
                                                  'Text'),
                                                 ('num', StandardScaler(),
                                                  ['Amount'])])),
                ('clf', LogisticRegression(max_iter=1000))])

In [9]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 1.0
               precision    recall  f1-score   support

entertainment       1.00      1.00      1.00         3
         food       1.00      1.00      1.00         7
     shopping       1.00      1.00      1.00         4
   technology       1.00      1.00      1.00         4
    transport       1.00      1.00      1.00         2

     accuracy                           1.00        20
    macro avg       1.00      1.00      1.00        20
 weighted avg       1.00      1.00      1.00        20



In [12]:
test_expenses = pd.DataFrame({
    "Text": [
        "movie ticket pvr",
        "ola cab ride",
        "netflix subscription",
        "bus ticket"
    ],
    "Amount": [300, 150, 499, 40]
})

model.predict(test_expenses)


array(['entertainment', 'food', 'entertainment', 'transport'],
      dtype=object)

In [11]:
df["Category"].value_counts()


food             30
transport        26
shopping         21
entertainment    12
technology       11
Name: Category, dtype: int64